# Introducción a los sistemas lineales



Gran parte de los problemas de la **ingeniería de telecomunicación** pueden formularse de una manera muy simple:

> tenemos **información**, esa información se representa mediante **señales**,
> y esas señales son **procesadas por sistemas**.

Este punto de vista permite analizar y diseñar soluciones en ámbitos muy distintos:

* transmisión por radio y fibra óptica,
* audio y vídeo digital,
* radar, GPS y sistemas de navegación,
* procesado de imagen y señal biomédica,
* control automático y electrónica.

Aunque los dispositivos físicos sean muy diferentes, las **herramientas matemáticas** que permiten describirlos son, en gran medida, las mismas.
Ese es precisamente el objetivo de esta asignatura: **proporcionar un marco común para estudiar señales y sistemas de forma rigurosa**.



## Señales y sistemas

Los conceptos de señales y sistemas surgen en gran variedad de campos. Tienen gran importancia en áreas tan diversas como: comunicaciones, aeronáutica, diseño de circuitos, acústica, ingeniería biomédica, ...

Aunque la naturaleza física de estas señales y sistemas pueda ser muy distinta, hay dos elementos comunes que permiten estudiarlas de forma conjunta:

### ¿Qué es una señal?
:::{important .simple icon=false}  Definición de Señal
Una **señal** es una función matemática que describe cómo varía una magnitud física (o informativa) respecto a una o varias variables independientes.
:::
De forma intuitiva:

* una señal **representa información**,
* esa información puede **medirse**, **almacenarse**, **transmitirse** o **procesarse**.

#### Ejemplos cotidianos

* La tensión eléctrica en un circuito en función del tiempo.
* La presión acústica de una señal de voz.
* La intensidad luminosa de cada píxel de una imagen.
* La temperatura medida por un sensor cada cierto intervalo de tiempo.

En todos los casos estamos describiendo un fenómeno mediante una **función**.

  ```{figure}  figures/T0/2_1_fig1
  ---
  width: 80%
  name: fig-senales
  ---
  Ejemplo de señal
  ```



### ¿Qué es un sistema?

:::{important .simple icon=false} Definición de Sistema
Un **sistema** es cualquier dispositivo, proceso o algoritmo que **transforma una señal de entrada en una señal de salida**.
:::

Desde el punto de vista del análisis, un sistema se estudiará como una **“caja negra”**:
*conocemos* qué señal entra, *observamos* qué señal sale y nos interesa *caracterizar* la relación entre ambas.

### Ejemplo

* **Sistema**: un circuito eléctrico.
* **Señales**: tensiones y corrientes $V(t)$, $I(t)$.

Este enfoque será clave cuando estudiemos los **sistemas lineales e invariantes en el tiempo (LTI)**, que constituyen el núcleo de la asignatura.


In [1]:
import numpy as np

from bokeh.plotting import figure, show, output_notebook
from bokeh.models import ColumnDataSource, CustomJS, Slider, Div, Select, Label
from bokeh.layouts import column, row

output_notebook(verbose=False, hide_banner=True);

# ==============================================================================
# 1. DATOS
# ==============================================================================

t = np.linspace(0, 6, 1200)

# Señal de entrada: seno + pulso rectangular
x = np.sin(2*np.pi*0.7*t) + 0.8*((t > 2.0) & (t < 3.2))

source = ColumnDataSource(data=dict(t=t, x=x, y=x))

# ==============================================================================
# 2. FIGURA
# ==============================================================================

p = figure(
    height=400,
    width=600,
    tools="pan,wheel_zoom,reset",
    title="Una señal de entrada es transformada por un sistema",
    sizing_mode="scale_width" 
)

p.line("t", "x", source=source, line_width=2, color="black",
       alpha=0.45, legend_label="Entrada x(t)")

p.line("t", "y", source=source, line_width=3, color="blue",
       legend_label="Salida y(t)")

p.xaxis.axis_label = "t"
p.yaxis.axis_label = "amplitud"
p.grid.grid_line_alpha = 0.25
p.legend.location = "top_right"
p.legend.click_policy = "hide"
p.toolbar.logo = None

# # Texto tipo caja negra
# block = Div(text="""
# <div style="
#     border: 2px solid #444;
#     border-radius: 10px;
#     padding: 15px;
#     width: 260px;
#     text-align: center;
#     font-family: sans-serif;
#     background-color: #f7f7f7;
# ">
#     <b>Entrada</b> &nbsp; x(t)
#     &nbsp; → &nbsp;
#     <span style="
#         border: 1px solid #333;
#         padding: 6px 12px;
#         border-radius: 6px;
#         background-color: white;
#     ">
#         Sistema
#     </span>
#     &nbsp; → &nbsp;
#     <b>Salida</b> &nbsp; y(t)
# </div>
# """)

# ==============================================================================
# 3. CONTROLES
# ==============================================================================

system_select = Select(
    title="Sistema",
    value="Amplificador",
    options=[
        "Amplificador",
        "Retardo",
        "Suavizado / filtro paso bajo",
        "Saturación no lineal"
    ],
    width=260
)

gain_slider = Slider(start=0.2, end=2.5, value=1.2, step=0.1,
                     title="Ganancia A", width=260)

delay_slider = Slider(start=0.0, end=1.5, value=0.5, step=0.05,
                      title="Retardo τ", width=260)

alpha_slider = Slider(start=0.02, end=0.5, value=0.12, step=0.02,
                      title="Suavizado α", width=260)

sat_slider = Slider(start=0.3, end=2.0, value=0.8, step=0.05,
                    title="Nivel de saturación", width=260)

info = Div(width=600)

# ==============================================================================
# 4. INTERACTIVIDAD
# ==============================================================================

callback = CustomJS(
    args=dict(
        source=source,
        system_select=system_select,
        gain_slider=gain_slider,
        delay_slider=delay_slider,
        alpha_slider=alpha_slider,
        sat_slider=sat_slider,
        info=info
    ),
    code="""
    const data = source.data;
    const t = data["t"];
    const x = data["x"];
    const y = data["y"];

    const system = system_select.value;
    const A = gain_slider.value;
    const tau = delay_slider.value;
    const alpha = alpha_slider.value;
    const sat = sat_slider.value;

    const dt = t[1] - t[0];

    if (system === "Amplificador") {
        for (let i = 0; i < x.length; i++) {
            y[i] = A * x[i];
        }

        info.text = `
        <div style="font-family:sans-serif; font-size:14px;">
        <b>Sistema amplificador:</b> la salida es una versión escalada de la entrada:
        <br>
        <span style="font-size:18px;">y(t) = A x(t)</span>
        <br><br>
        Este sistema es <b>lineal</b>: si duplicamos la entrada, se duplica la salida.
        </div>`;
    }

    else if (system === "Retardo") {
        const shift = Math.round(tau / dt);

        for (let i = 0; i < x.length; i++) {
            if (i - shift >= 0) {
                y[i] = x[i - shift];
            } else {
                y[i] = 0;
            }
        }

        info.text = `
        <div style="font-family:sans-serif; font-size:14px;">
        <b>Sistema con retardo:</b> la salida reproduce la entrada, pero desplazada en el tiempo:
        <br>
        <span style="font-size:18px;">y(t) = x(t - τ)</span>
        <br><br>
        Este sistema no cambia la forma de la señal, solo retrasa cuándo aparece.
        </div>`;
    }

    else if (system === "Suavizado / filtro paso bajo") {
        y[0] = x[0];

        for (let i = 1; i < x.length; i++) {
            y[i] = alpha * x[i] + (1 - alpha) * y[i-1];
        }

        info.text = `
        <div style="font-family:sans-serif; font-size:14px;">
        <b>Filtro paso bajo sencillo:</b> la salida se calcula mezclando la entrada actual
        con la salida anterior:
        <br>
        <span style="font-size:18px;">y[n] = αx[n] + (1-α)y[n-1]</span>
        <br><br>
        El sistema suaviza cambios bruscos y elimina parte de las variaciones rápidas.
        </div>`;
    }

    else if (system === "Saturación no lineal") {
        for (let i = 0; i < x.length; i++) {
            if (x[i] > sat) {
                y[i] = sat;
            } else if (x[i] < -sat) {
                y[i] = -sat;
            } else {
                y[i] = x[i];
            }
        }

        info.text = `
        <div style="font-family:sans-serif; font-size:14px;">
        <b>Sistema con saturación:</b> la salida no puede superar un cierto valor máximo:
        <br>
        <span style="font-size:18px;">y(t) = clip(x(t))</span>
        <br><br>
        Este sistema es <b>no lineal</b>: si aumentamos mucho la entrada, la salida deja de crecer.
        </div>`;
    }

    source.change.emit();
    """
)

for w in [system_select, gain_slider, delay_slider, alpha_slider, sat_slider]:
    w.js_on_change("value", callback)

callback.args["source"].data["y"] = x.copy()

# Ejecutar una vez al cargar
info.text = """
<div style="font-family:sans-serif; font-size:14px;">
<b>Sistema amplificador:</b> la salida es una versión escalada de la entrada:
<br>
<span style="font-size:18px;">y(t) = A x(t)</span>
<br><br>
Este sistema es <b>lineal</b>: si duplicamos la entrada, se duplica la salida.
</div>
"""

controls = column(system_select, gain_slider, delay_slider, alpha_slider, sat_slider)

layout = column(
    p, 
    controls,
    info
)

show(layout)



## Problemas de procesado de señales

* **Análisis**: estudiar la respuesta de un sistema específico a diversas entradas. (Convolución).
  Ejemplo: análisis de circuitos.

  ```{figure}  figures/T0/1_fig3 
  ---
  width: 60%
  name: fig-analisis
  ---
  ```

* **Diseño o identificación**: diseñar sistemas para procesar señales de determinada forma.
  Ejemplos: restauración (voz, imagen, ...), realce, extracción de características.

  ```{figure}  figures/T0/1_fig4 
  ---
  width: 60%
  ---
  ```

* **Deconvolución**: obtener entrada para un sistema dado a partir de su salida.
  Ejemplos: eliminar aberraciones en lentes de cámaras fotográficas o movimiento.

  ```{figure}  figures/T0/1_fig5 
  ---
  width: 60%
  ---
  ```

* **Filtrado**: obtener el sistema y la señal de salida que permite modificar una señal de entrada de determinada forma.
  Ejemplo: eliminar altas frecuencias de señal musical.

  ```{figure}  figures/T0/1_fig6 
  ---
  width: 60%
  ---
  ```

* **Modelado**: diseñar un sistema y la señal de entrada que nos permite obtener una salida determinada.
  Ejemplo: sintetizar voz.

  ```{figure}  figures/T0/1_fig7 
  ---
  width: 60%
  ---
  ```

* **Control**: diseñar un sistema que controle a otro a partir de su salida.
  Ejemplos: sistema de control de planta química, piloto automático.

  ```{figure}  figures/T0/1_fig8 
  ---
  width: 60%
  ---
  ```



<!-- 

## Clases de señales

**Tipos de señales**:

* Según el rango de variabilidad de la variable independiente: continua vs. discreta.

  ```{figure}  figures/T0/1_fig9 
  ---
  width: 80%
  ---
  ```

  * **Continua**: valores para todos los puntos del eje de abscisas. Ejemplo: señales en circuitos eléctricos y mecánicos.
  * **Discreta**: valores en puntos discretos y equiespaciados del eje de abscisas. Ejemplo: promedio de la bolsa cada día.

* Según el rango de variabilidad de la variable dependiente: analógica vs. digital.

  ```{figure}  figures/T0/1_fig10 
  ---
  width: 80%
  ---
  ```

  * **Analógica**: puede tomar cualquier valor dentro de un rango. Ejemplo: temperatura.
  * **Digital**: puede tomar sólo valores cuantizados. Ejemplo: luz encendida o apagada.

* Según el número de variables independientes: unidimensional vs. multidimensional.

  ```{figure}  figures/T0/1_fig11 
  ---
  width: 80%
  ---
  ```

* Según la incertidumbre de la variable dependiente: señal determinista vs. aleatoria.

  * **Determinista**: se conocen los valores que toma en todos y cada uno de sus instantes.
  * **Aleatoria o estocástica**: hay incertidumbre sobre el valor que toma en alguno de sus instantes. Asociado al concepto de probabilidad.

```{math}
\text{Señal} \begin{cases}
\text{Determinista}
  \begin{cases}
    \text{Estacionaria}
      \begin{cases}
        \text{Periódica}\\
        \text{No periódica}
      \end{cases}\\
    \text{No Estacionaria}
  \end{cases}\\
\text{Aleatoria o Estocástica}
  \begin{cases}
    \text{Estacionaria}
      \begin{cases}
        \text{Ergódica}\\
        \text{No ergódica}
      \end{cases}\\
    \text{No estacionaria}
  \end{cases}
\end{cases}
```

**Tipos de sistemas**: continuos, discretos, analógicos, digitales.

---

## Ejemplos de señales y sistemas

Ejemplos de señales:

* Habla: telefonía, radio, ..., vida cotidiana.
* Señales biomédicas:

  * 1-D: encefalograma, electrocardiograma.
  * 2-D: radiografía, angiografía, ecografía.
  * 3-D: TAC, RM, ultrasonidos 3D, vídeo.
  * 4-D: secuencias temporales de volúmenes.
  * N-D: volúmenes con datos tensoriales para estudiar fibras nerviosas del cerebro.
* Sonido y música.
* Vídeo e imagen.
* Señales de radar, sonar, satélite.
* Comunicación de datos entre ordenadores.

Otros ejemplos:

* Circuitos eléctricos:

  * Señales: $I(t), V(t)$.
  * Sistema: propio circuito.
* Automóvil:

  * Señales: entrada: presión en pedales y giro del volante. Salida: aceleración y dirección del automóvil.
  * Sistema: automóvil.
* Compresión de imagen:

  * Señales: imagen comprimida y sin comprimir.
  * Sistemas: compresor y descompresor.
* Transmisión de señal de radio:

  * Señal: señal de audio modulada.
  * Sistema: medio de transmisión (atmósfera).

Ejemplos de procesado de señal:

* Eliminación de ruido en voz de piloto de avión (comunicaciones).
* Realce de fotografías.
* Extracción de parámetros de interés:

  * Formantes de la voz (identificación).
  * Reconocimiento de habla.
  * Identificación de formas (visión artificial).
 -->
